In [ ]:
from numpy.linalg import norm
from rulebook_benchmark.rulebook import Rulebook, Rule
from rulebook_benchmark.runner import run_scenic_metadrive
from rulebook_benchmark.plotting import animate_trajectory_with_violations

## Define Simple Rules

This section defines a few basic rules for evaluation, using these heuristics to calculate violation each step:

- Count collisions with other vehicles
- Count only new collisions using the built-in collision_timeline method
- Count nearby vehicles within a distance threshold
- Register when ego speed exceeds a limit

In [ ]:
def simple_vehicle_collision(handler, step): # simple collision rule that punishes the number of collisions with other vehicles at each step
    pool = handler(step)
    return len(pool.vehicles_colliding) # each pool and vehicles_colliding is cached, so they will not be recomputed if called multiple times over many rules
    # alternatively, we can directly compute the collisions by using the built-in colliding() method, which is not cached    
    vehicle_states = pool.vehicle_states
    return len(pool.colliding(vehicle_states))

def simple_vehicle_collision_with_timeline(handler,step):
    collision_timeline = handler.collision_timeline # stores (start, end) of each collision in the form timeline[uid] = [(start1, end1), (start2, end2), ...]
    # collision_timeline is cached, so it will not be recomputed if called multiple times over many rules
    # to avoid punishing the same collision multiple times, we can directly punish new collisions at each step
    vehicle_uids = handler.vehicle_uids
    new_collisions = 0
    for uid in vehicle_uids:
        uid_collisions = collision_timeline.get(uid, [])
        for (start, end) in uid_collisions:
            if start == step: # new collision starting at this step
                new_collisions += 1
    
    return new_collisions


def simple_vehicle_clearance(handler, step, threshold): # simple clearance rule that punishes the number of vehicles within a certain distance threshold at each step
    pool = handler(step)
    return len(pool.in_proximity(pool.ego_state, pool.vehicle_states, threshold))




def simple_speed_limit(handler, step, speed_limit): # simple speed limit rule that punishes the ego vehicle if its speed exceeds the speed limit at each step
    pool = handler(step)
    ego_velocity = pool.ego_state.velocity
    ego_speed = norm(ego_velocity)
    return 1 if ego_speed > speed_limit else 0

## Create Rulebook

This cell wraps the rule functions into Rule objects and builds two rulebooks:

- One using direct collision counts
- One using timeline-based collision counts

Both use the same graph file and can be evaluated on the same realization for comparison.

In [ ]:
collision_rule = Rule(simple_vehicle_collision, aggregation_method=sum, name="simple_vehicle_collision", rule_id=1)
collision_timeline_rule = Rule(simple_vehicle_collision_with_timeline, aggregation_method=sum, name="simple_vehicle_collision_with_timeline", rule_id=1)
clearance_rule = Rule(simple_vehicle_clearance, aggregation_method=sum, name="simple_vehicle_clearance", rule_id=2, threshold=2.0)
speed_limit_rule = Rule(simple_speed_limit, aggregation_method=sum, name="simple_speed_limit", rule_id=3, speed_limit = 2.0)

rule_id_to_rule = {collision_rule.id: collision_rule, clearance_rule.id: clearance_rule, speed_limit_rule.id: speed_limit_rule}
rule_id_to_rule_2 = {collision_timeline_rule.id: collision_timeline_rule, clearance_rule.id: clearance_rule, speed_limit_rule.id: speed_limit_rule}
rulebook = Rulebook(rule_id_to_rule=rule_id_to_rule, rulebook_file="tutorial_rulebook.graph")
rulebook_2 = Rulebook(rule_id_to_rule=rule_id_to_rule_2, rulebook_file="tutorial_rulebook.graph")

## Run Scenario

Runs the Scenic simulation and stores the resulting trajectory data in `realization`, which is used in later evaluation and visualization steps.

In [ ]:
from rulebook_benchmark.runner import run_scenic_metadrive
realization = run_scenic_metadrive(scenic_path = "tutorial.scenic", max_steps=100, render=False, seed=1)

## Evaluate Rules

Runs both rulebooks on the generated realization and stores their violation results for comparison.

In [ ]:
results = rulebook.evaluate(realization)
results_2 = rulebook_2.evaluate(realization)

In [ ]:
animation = animate_trajectory_with_violations(realization, rulebook, results, margin=50, buffer=100)
animation_2 = animate_trajectory_with_violations(realization, rulebook_2, results_2)

In [ ]:
# play the animation in a Jupyter notebook
from IPython.display import HTML
HTML(animation.to_jshtml())

In [ ]:
from IPython.display import HTML
HTML(animation_2.to_jshtml())

In [ ]:
# Alternatively, you can define a rulebook in another file, then import it

from tutorial_rules import example_rulebook
results_3 = example_rulebook.evaluate(realization)
animation_3 = animate_trajectory_with_violations(realization, example_rulebook, results_3)


In [ ]:
HTML(animation_3.to_jshtml())